<a href="https://colab.research.google.com/github/marchedev2002/ia/blob/main/clasificador_nacho/Clasificador_Frutas_Nacho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🍎 Clasificación de Frutas con ResNet18 y Transfer Learning
## Universidad Tecnológica Nacional – Regional Rosario
### Materia: Inteligencia Artificial | PyTorch 2.x
**Alumno**: Jose Ignacio Dayer (Nacho)

---

Este notebook implementa un clasificador de frutas utilizando **Transfer Learning** con una arquitectura **ResNet18** preentrenada en ImageNet, con entrenamiento en dos fases (Warm-up y Fine-Tuning) y **Test-Time Augmentation (TTA)** para mejorar la generalización sobre el test set realista.


## Celda 0: Configuración del entorno y chequeo de GPU


In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No se detectó GPU. En Colab: Entorno de ejecución → Cambiar tipo → GPU T4')


## Paso 1: Importar librerías necesarias


In [ ]:
import os
import sys
import time
import random
import warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅ Librerías importadas e inicializadas")


## Paso 2: Descarga del Dataset y Test Set de la Competencia (Solo en Colab)

Si estás en Colab, esta celda descargará automáticamente el dataset de entrenamiento y las imágenes de prueba utilizando la API de Kaggle con tus credenciales.


In [ ]:
# Detectar entorno automáticamente
IS_KAGGLE = os.path.exists("/kaggle/input")

if not IS_KAGGLE:
    # Configurar token de la API de Kaggle para Colab
    os.environ["KAGGLE_API_TOKEN"] = "KGAT_c8dc4e2dc1eda9c2becc05bd31bb1e8c"
    
    # Descargar datasets de entrenamiento y prueba
    print("📥 Descargando dataset de entrenamiento (Food Classification)...")
    !kaggle datasets download -d geronimoforconi/utn-ia-2026-food-classification-training-set -p /content/data/fruit_recognition --unzip -q
    
    print("📥 Descargando test set de la competencia...")
    !mkdir -p /content/data
    !kaggle competitions download -c utn-ia-2026-fruit-vegetable-classifier -p /content/data
    !mv /content/data/utn-ia-2026-fruit-vegetable-classifier.zip /content/data/utn_competencia_test.zip
    !mkdir -p /content/data/test_competencia
    !unzip -q /content/data/utn_competencia_test.zip -d /content/data/test_competencia
    print("✅ Descargas completas en /content/data")
else:
    print("Kaggle Environment detectado. No es necesario descargar nada manualmente.")


## Paso 3: Configuración de rutas y partición (Split)

Definimos las rutas correspondientes y, si estamos en Colab, realizamos el split de carpetas para entrenamiento y validación.


In [ ]:
import shutil

if IS_KAGGLE:
    COMP_DIR   = Path("/kaggle/input/utn-ia-2026-fruit-vegetable-classifier")
    TRAIN_DIR  = Path("/kaggle/input/utn-ia-2026-food-classification/train")
    TEST_DIR   = COMP_DIR / "test_images"
    SAMPLE_SUB = COMP_DIR / "sample_submission.csv"
    OUTPUT_DIR = Path("/kaggle/working")
else:
    COMP_DIR   = Path("/content/data/test_competencia/utn_competencia_test")
    TRAIN_DIR  = Path("/content/data/fruit_recognition_splint/train")
    TEST_DIR   = Path("/content/data/test_competencia/utn_competencia_test/test_images")
    SAMPLE_SUB = Path("/content/data/test_competencia/utn_competencia_test/sample_submission.csv")
    OUTPUT_DIR = Path("/content")

CLASSES = ["apple", "banana", "grapes", "potato", "tomato"]

# Si estamos en Colab, creamos el split de carpetas de validación y test
if not IS_KAGGLE:
    print("✂️ Creando subset de validación y test...")
    
    def crear_subset_desde_train(clases, raiz, destino, split_train=0.70, split_val=0.15):
        for clase in clases:
            imgs = sorted([
                p for p in (raiz / clase).glob('**/*')
                if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
            ])
            random.shuffle(imgs)
            n = len(imgs)
            n_val  = int(n * split_val)
            n_test = int(n * (1 - split_train - split_val))
            
            splits = {
                'val':   imgs[:n_val],
                'test':  imgs[n_val:n_val + n_test],
                'train': imgs[n_val + n_test:],
            }
            for split_name, subset in splits.items():
                dst = destino / split_name / clase
                dst.mkdir(parents=True, exist_ok=True)
                for img in subset:
                    shutil.copy(img, dst / img.name)
    
    # Limpiar split previo si existe
    if Path('/content/data/fruit_recognition_splint').exists():
        shutil.rmtree('/content/data/fruit_recognition_splint')
        
    crear_subset_desde_train(
        CLASSES, 
        Path('/content/data/fruit_recognition'), 
        Path('/content/data/fruit_recognition_splint'), 
        0.70, 0.15
    )
    print("✅ Split de carpetas creado exitosamente en /content/data/fruit_recognition_splint")


## Paso 4: Transformaciones y Data Augmentation

Definimos las transformaciones de datos utilizando torchvision.


In [ ]:
# Parámetros estándar de ImageNet
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Pipeline de entrenamiento
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomApply([transforms.RandomPosterize(bits=3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15))
])

# Pipeline de validación y test
transform_val_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

def cargar_imagen_limpia(path):
    """Limpia transparencias de la imagen convirtiendo a RGB sobre fondo blanco."""
    img = Image.open(path)
    if img.mode in ('P', 'RGBA', 'LA'):
        img = img.convert('RGBA')
        fondo = Image.new('RGB', img.size, (255, 255, 255))
        fondo.paste(img, mask=img.split()[3])
        return fondo
    return img.convert('RGB')

# Cargar datasets con ImageFolder
train_dataset = ImageFolder(str(TRAIN_DIR), transform=transform_train, loader=cargar_imagen_limpia)
val_dataset   = ImageFolder(str(TRAIN_DIR.parent / 'val'), transform=transform_val_test, loader=cargar_imagen_limpia)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

class_names_r = train_dataset.classes
print(f"📊 Dataset cargado: train={len(train_dataset)} | val={len(val_dataset)}")
print(f"Clases: {class_names_r}")


## Paso 5: Definición del Modelo ResNet18

Cargamos una **ResNet18** preentrenada, congelamos sus pesos y reemplazamos la capa final `fc` para adaptarla a nuestras 5 clases.


In [ ]:
def crear_resnet(num_classes):
    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)
    
    # Congelar todas las capas
    for param in model.parameters():
        param.requires_grad = False
        
    # Reemplazar capa final
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def descongelar_ultimas_capas(model):
    # Descongelar layer4 y fc para fine-tuning
    for name, param in model.named_parameters():
        if "layer4" in name or "fc" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

model = crear_resnet(len(CLASSES)).to(device)
criterion = nn.CrossEntropyLoss()
print("✅ Modelo cargado con éxito.")


## Paso 6: Ciclos de Entrenamiento y Evaluación


In [ ]:
def entrenar_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        
        _, preds = out.max(1)
        total_loss += loss.item() * imgs.size(0)
        correct += preds.eq(labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def evaluar(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            
            _, preds = out.max(1)
            total_loss += loss.item() * imgs.size(0)
            correct += preds.eq(labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total


## Paso 7: Fase 1 — Entrenando solo el clasificador final (fc)

Entrenamos la capa `fc` durante 20 épocas con Adam. Mantenemos el extractor de características congelado.


In [ ]:
EPOCHS_P1 = 20
optimizer_p1 = optim.Adam(model.fc.parameters(), lr=1e-3)
best_val_loss = float('inf')
ckpt_path_p1 = OUTPUT_DIR / "resnet18_mejor_realista.pth"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9}")
print("-" * 58)

for epoch in range(EPOCHS_P1):
    tr_loss, tr_acc = entrenar_epoch(model, train_loader, criterion, optimizer_p1, device)
    val_loss, val_acc = evaluar(model, val_loader, criterion, device)
    
    mark = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc
        }, ckpt_path_p1)
        mark = " ✅"
        
    print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>9.1%} | {val_loss:>10.4f} | {val_acc:>9.1%}{mark}")


## Paso 8: Fase 2 — Fine-Tuning (Descongelando layer4 y fc)

Ahora descongelamos la capa convolucional superior (`layer4`) y el clasificador `fc`. Usamos un optimizador `AdamW` con tasas de aprendizaje diferenciales y regularización por decaimiento de pesos, además de un programador de tasa de aprendizaje `ReduceLROnPlateau` y early stopping.


In [ ]:
# Cargar el mejor modelo de la Fase 1
checkpoint = torch.load(ckpt_path_p1, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"🏆 Cargado mejor modelo de la Fase 1 (Val Acc: {checkpoint['val_acc']:.2%})")

# Descongelar las capas correspondientes
descongelar_ultimas_capas(model)

optimizer_ft = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(),     'lr': 1e-4}
], weight_decay=1e-2)

scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ft, mode='min', factor=0.5, patience=2
)

EPOCHS_FT = 20
PACIENCIA_ES = 4
best_val_loss_ft = float('inf')
epochs_no_improve = 0
ckpt_path_ft = OUTPUT_DIR / "resnet18_finetuned.pth"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | lr")
print("-" * 65)

for epoch in range(EPOCHS_FT):
    tr_loss, tr_acc = entrenar_epoch(model, train_loader, criterion, optimizer_ft, device)
    val_loss, val_acc = evaluar(model, val_loader, criterion, device)
    scheduler_ft.step(val_loss)
    
    mark = ""
    if val_loss < best_val_loss_ft:
        best_val_loss_ft = val_loss
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc
        }, ckpt_path_ft)
        mark = " ✅"
    else:
        epochs_no_improve += 1
        mark = ""
        
    lr_actual = optimizer_ft.param_groups[0]['lr']
    print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>9.1%} | {val_loss:>10.4f} | {val_acc:>9.1%}{mark} | {lr_actual:.1e}")
    
    if epochs_no_improve >= PACIENCIA_ES:
        print(f"⏹ Early stopping en epoch {epoch} (sin mejoras en {PACIENCIA_ES} épocas)")
        break


## Paso 9: Generación de submission.csv con Test Time Augmentation (TTA)

Cargamos el mejor modelo de Fine-Tuning y predecimos las imágenes del Test Set aplicando TTA. Esto promedia las probabilidades de salida de la imagen original, su espejo horizontal y dos rotaciones (10° y -10°) para dar una predicción final más robusta.


In [ ]:
# Cargar el mejor modelo fine-tuneado
checkpoint_ft = torch.load(ckpt_path_ft, map_location=device)
model.load_state_dict(checkpoint_ft['model_state_dict'])
model.eval()
print(f"🏆 Cargado mejor modelo fine-tuneado (Val Acc: {checkpoint_ft['val_acc']:.2%})")

def predecir_imagen_con_tta(model, img_path, transform, device):
    img = cargar_imagen_limpia(img_path)
    
    # 4 vistas de la imagen para TTA
    vistas = [
        img,
        img.transpose(Image.FLIP_LEFT_RIGHT),
        img.rotate(10),
        img.rotate(-10)
    ]
    
    with torch.no_grad():
        probs = []
        for v in vistas:
            t = transform(v).unsqueeze(0).to(device)
            logits = model(t)
            probs.append(F.softmax(logits, dim=1))
        
        prob_promedio = torch.mean(torch.stack(probs), dim=0)
        return prob_promedio.cpu().numpy()[0]

# Leer el sample submission para las IDs
sample_sub = pd.read_csv(SAMPLE_SUB)
image_ids = sample_sub["image_id"].tolist()

print(f"🔄 Procesando {len(image_ids)} imágenes con TTA...")
predicciones = []

for img_id in image_ids:
    img_path = None
    for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
        p = TEST_DIR / (img_id + ext)
        if p.exists():
            img_path = p
            break
    if img_path is None:
        img_path = TEST_DIR / img_id
        
    probs_img = predecir_imagen_con_tta(model, img_path, transform_val_test, device)
    pred_idx = probs_img.argmax()
    pred_label = class_names_r[pred_idx]
    
    predicciones.append({
        "image_id": img_id,
        "label": pred_label
    })

df_sub = pd.DataFrame(predicciones)
print("✅ Predicciones generadas!")

# Validar y guardar csv
df_sub.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print("📄 submission.csv guardado con éxito!")
print(df_sub.head(10))
print("
Distribución de predicciones:")
print(df_sub["label"].value_counts())
